# Foundation Models Sandbox: Comparing CausalPFN, Do-PFN & CausalFM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chris-L6/causalfm-survey/blob/main/notebooks/Foundation_models_sandbox.ipynb)

**This is a sandbox notebook for comparing all three causal foundation
models side by side on the same dataset** (CausalPFN, Do-PFN, and CausalFM),
each run through its own native API, with results and a comparison plot at
the end.

If you just want to get one model running as fast as possible, see
[`Foundation_models_quickstart.ipynb`](Foundation_models_quickstart.ipynb),
which walks through CausalPFN alone end to end.

This notebook is standalone, it doesn't import this repo's `causal_bench`
wrappers. Every call below is exactly what you'd write reading each model's own
README, so you can lift a cell straight into your own project.

| Step | What it does |
|---|---|
| §0 | One-time environment check — **run this first** if you want Do-PFN to work |
| §1 | Simulate one dataset with a known (but normally unobservable) treatment effect |
| §2 | Run **CausalPFN** — `pip install causalpfn`, native `CATEEstimator` / `ATEEstimator` |
| §3 | Run **Do-PFN** — `git clone`, native `DoPFNRegressor` |
| §4 | Run **CausalFM** — `git clone` + checkpoint, native `StandardCATEModel` |
| §5 | Compare all three against ground truth |

## Why these aren't just "another sklearn model"

CausalPFN, Do-PFN and CausalFM are **amortized** / **in-context** estimators: a single
transformer is pretrained once (by the model authors, on millions of synthetic
causal-inference problems) and shipped as frozen weights. There is no
per-dataset training loop on your end — `fit()` just packages your data as
"context" for a forward pass.

```python
# Traditional metalearner (e.g. T-learner): trains fresh parameters on YOUR data
model_treated = RandomForest().fit(X[T == 1], Y[T == 1])
model_control = RandomForest().fit(X[T == 0], Y[T == 0])
tau_hat = model_treated.predict(X_test) - model_control.predict(X_test)

# Causal foundation model: weights are already trained; "fit" just stores context
cate_estimator = CATEEstimator(device=device)     # pretrained weights, downloaded once
cate_estimator.fit(X_train, T_train, Y_train)      # NOT gradient descent on your data
tau_hat = cate_estimator.estimate_cate(X_test)     # one forward pass, conditioned on context
```

**Practical upshot:** `fit()` is cheap, and the same frozen network is reused
across every dataset — that's what "zero-shot" means here. The three models
below differ mainly in how you get their weights (PyPI vs. `git clone`) and
the exact shape of their `fit`/`predict` calls.

## 0. One-time environment check — run this cell FIRST

**Only needed if you want Do-PFN to work** (§3). Its model code depends on an
internal PyTorch name that PyTorch removed in `torch>=2.10` (present through
`v2.9.0`, gone in `v2.10.0`).

- **On Colab**: pins `torch==2.9.1` (verified compatible with Do-PFN) via a
  plain `pip install` -- **can take up to a minute or so** (torch is a large
  package; network speed is the bottleneck, not this notebook). As long as
  this is the first cell you run, **no restart needed** -- Colab ships torch
  preinstalled but never auto-imports it, so nothing has loaded the
  incompatible version into memory yet; the freshly installed one is just
  what later cells pick up. (If you'd already run other cells before this
  one, torch may already be in memory -- restart the runtime and re-run from
  the top.)
- **Locally, in this repo's `uv` venv**: `pip` isn't available inside the
  notebook here, so this cell can only detect the problem, not fix it. Run in
  a terminal instead: `uv pip install "torch==2.9.1"`, then restart the
  kernel and re-run from the top.
- **Not planning to run Do-PFN?** Skip this — CausalPFN and CausalFM both work
  fine on any recent torch.

In [ ]:
import sys, importlib.metadata

IN_COLAB = "google.colab" in sys.modules
TORCH_PIN = "2.9.1"  # last version verified compatible with Do-PFN (torch>=2.10 breaks it)

def _torch_needs_downgrade():
    try:
        v = importlib.metadata.version("torch")  # reads metadata, doesn't import torch
    except importlib.metadata.PackageNotFoundError:
        return False  # not installed yet -- nothing to fix here
    major, minor = (int(p) for p in v.split("+")[0].split(".")[:2])
    return (major, minor) >= (2, 10)

if not _torch_needs_downgrade():
    print("OK -- torch version is compatible with Do-PFN (or not installed yet).")
elif IN_COLAB:
    print(f"torch >= 2.10 detected -- installing torch=={TORCH_PIN} (can take a minute)...")
    get_ipython().system(f"pip install -q torch=={TORCH_PIN}")
    print("OK -- done. If this was the first cell you ran, no restart needed -- "
          "continue to the next cell.")
else:
    print("torch >= 2.10 detected -- Do-PFN (section 3) will fail to import.")
    print('Fix, in a terminal (not this notebook -- local uv venv has no pip):')
    print(f'    uv pip install "torch=={TORCH_PIN}"')
    print("then restart this notebook's kernel and re-run from the top.")

## 1. Example dataset — a simulated discount-email campaign

Rather than an abstract `X0, X1, ...` matrix, we simulate a small business
scenario in the spirit of Facure's *Causal Inference for the Brave and True*:
an online retailer sends a discount email to a subset of customers and wants
to know its effect on next-month spend.

- **Covariates**: `recency` (days since last purchase, standardized — higher
  means more lapsed), `monetary` (average past order value, standardized),
  `age` (standardized).
- **Treatment** `T`: received the discount email. It's **confounded on
  purpose** — marketing targets loyal, high-spend, recently-active customers,
  so a naive treated-vs-untreated comparison is biased.
- **Outcome** `Y`: next-month spend.
- **Ground truth** (known only because this is simulated, never in real data):
  the email works *better* on lapsed customers and *worse* on older ones —
  `tau(x) = 2.0 + 1.5 * recency - 0.75 * age`, a heterogeneous effect that lets
  us score each model's CATE estimate, not just its ATE.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)
n = 1500

recency = rng.normal(0, 1, n)    # standardized days since last purchase
monetary = rng.normal(0, 1, n)   # standardized average past order value
age = rng.normal(0, 1, n)        # standardized customer age
X = np.column_stack([recency, monetary, age]).astype(np.float32)

# Confounded treatment assignment: loyal, high-spend, recently-active
# customers are more likely to be targeted with the discount email.
propensity = 1 / (1 + np.exp(-(0.8 * monetary - 0.6 * recency)))
T = rng.binomial(1, propensity).astype(np.float32)

# True heterogeneous treatment effect (unobservable outside a simulation).
tau_true = (2.0 + 1.5 * recency - 0.75 * age).astype(np.float32)

# Potential outcomes -> observed outcome.
noise = rng.normal(0, 1.0, n).astype(np.float32)
Y0 = (5.0 + 2.0 * monetary - 0.5 * age + noise).astype(np.float32)
Y1 = Y0 + tau_true
Y = np.where(T == 1, Y1, Y0).astype(np.float32)

X_train, X_test, T_train, T_test, Y_train, Y_test, tau_train, tau_test = train_test_split(
    X, T, Y, tau_true, test_size=0.3, random_state=SEED
)
true_ate = float(tau_true.mean())

print(f"n_train / n_test               : {len(X_train)} / {len(X_test)}")
print(f"Naive treated-vs-control gap   : {Y[T == 1].mean() - Y[T == 0].mean():.3f}  (biased by confounding)")
print(f"True ATE (known only here)     : {true_ate:.3f}")

In [ ]:
# Tiny scoring helpers -- kept inline since this notebook has no other deps.
import os, sys, time

def pehe(tau_hat, tau_ref):
    return float(np.sqrt(np.mean((np.asarray(tau_hat) - np.asarray(tau_ref)) ** 2)))

def ate_abs_error(ate_hat, ate_ref):
    return float(abs(ate_hat - ate_ref))

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

IN_COLAB = "google.colab" in sys.modules
# NOTE (local `uv` venvs only): `%pip install` / `!pip install` cells below
# silently no-op locally ("No module named pip") -- they only work on Colab,
# which ships pip. Locally, install missing packages with `uv pip install <pkg>`.

results = {}  # model name -> dict(tau_hat, ate_hat, runtime, pehe, ate_abs_error)

## 2. CausalPFN

**Install:** `pip install causalpfn` — a normal PyPI package. The first call
downloads pretrained weights from the Hugging Face Hub (a few hundred MB), so
it needs internet access once; after that they're cached locally.

**Caveat:** on Apple Silicon macOS, CausalPFN segfaults — a hard process
crash, not something a Python `try/except` can catch — on both CPU and MPS.
The likely cause is an unstable `scaled_dot_product_attention` kernel on
macOS's backends, not a hard CUDA requirement, so this cell proactively skips
on that specific combination rather than crashing the kernel. It's expected to
run fine on Colab, on either GPU or CPU.

The call below is CausalPFN's own two-estimator API, unmodified — a
`CATEEstimator` for the per-unit effect and a separate `ATEEstimator` for the
population average:

In [ ]:
import importlib.util
if importlib.util.find_spec("causalpfn") is None:
    %pip install -q causalpfn
import platform

# Not a Python exception, so check before calling rather than try/except.
APPLE_SILICON_MACOS = platform.system() == "Darwin" and platform.machine() == "arm64"

if device != "cuda" and APPLE_SILICON_MACOS:
    print("⚠ Skipping CausalPFN: segfaults on Apple Silicon macOS (CPU and MPS) "
          "-- should be fine on Colab, GPU or CPU.")
else:
    try:
        from causalpfn import CATEEstimator, ATEEstimator
    except ImportError:
        print("✗ causalpfn not installed -- run the %pip install line above.")
    else:
        t0 = time.time()

        cate_estimator = CATEEstimator(device=device, verbose=False)
        cate_estimator.fit(X_train, T_train, Y_train)
        tau_hat = np.asarray(cate_estimator.estimate_cate(X_test)).reshape(-1)

        ate_estimator = ATEEstimator(device=device, verbose=False)
        ate_estimator.fit(X_train, T_train, Y_train)
        ate_hat = float(np.asarray(ate_estimator.estimate_ate()).reshape(-1)[0])

        runtime = time.time() - t0
        results["CausalPFN"] = dict(
            tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
            pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
        )
        print(f"✓ CausalPFN | PEHE={results['CausalPFN']['pehe']:.3f}  "
              f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")

## 3. Do-PFN

Do-PFN isn't on PyPI, so this cell clones the repo directly. Skip its bundled
`requirements.txt` — it's a full research environment (pins `catboost==1.1.1`,
which has no wheel for recent Python, purely for baseline comparisons
`DoPFNRegressor` doesn't even use). `networkx`, `tqdm`, and `einops` on top of
what's already installed above is all it actually needs.

A few details found by reading the current source rather than the README,
since the two have drifted apart:

1. **Import from `scripts.transformer_prediction_interface`**, not `dopfn` —
   the module layout has changed since the README was written.
2. **Treatment goes in column 0** of the input matrix, not the last column —
   `predict_cid` overwrites `X[:, 0]` internally.
3. **The checkpoint loads via a path relative to the repo root**, so the
   working directory needs to be `Do-PFN/` while constructing and fitting the
   model.

It exposes a dedicated `predict_cate(X)` method — no need to call
`predict_full` twice and subtract by hand — but it expects a `torch.Tensor`,
not a numpy array.

If you hit `cannot import name 'Optional' from 'torch.nn.modules.transformer'`,
that's a torch-version mismatch — run the "0. One-time environment check"
cell above.

In [ ]:
import subprocess, warnings

DOPFN_DIR = "Do-PFN"
DOPFN_URL = "https://github.com/jr2021/Do-PFN.git"

if not os.path.exists(DOPFN_DIR):
    print(f"Cloning {DOPFN_URL} ...")
    subprocess.run(["git", "clone", DOPFN_URL], check=True)
sys.path.insert(0, os.path.abspath(DOPFN_DIR))

# NOT `pip install -r Do-PFN/requirements.txt` -- see markdown above.
if IN_COLAB:
    get_ipython().system("pip install -q networkx tqdm einops")
# Locally: uv pip install networkx tqdm einops

# Import through fit/predict is wrapped in one try/except: if this cell
# already failed once in this session (e.g. torch >= 2.10), Python can leave
# a partial import cached and a retry may silently "succeed" only to fail
# later inside fit()/predict_cate() instead. If you see the message below,
# run "0." above, restart the kernel, then re-run all cells from the top.
try:
    from scripts.transformer_prediction_interface import DoPFNRegressor

    # Treatment in COLUMN 0 -- see gotcha #2 above.
    X_full_train = np.concatenate([T_train.reshape(-1, 1), X_train], axis=1)
    X_full_test = np.concatenate(
        [np.zeros((len(X_test), 1), dtype=np.float32), X_test], axis=1
    )  # column 0 here is a placeholder; predict_cate overwrites it internally

    t0 = time.time()
    _cwd = os.getcwd()
    os.chdir(DOPFN_DIR)  # relative checkpoint path -- see gotcha #3 above
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            dopfn = DoPFNRegressor()
            dopfn.show_progress = False  # suppress its internal tqdm bars
            dopfn.fit(X_full_train, Y_train)
            tau_hat = np.asarray(dopfn.predict_cate(torch.as_tensor(X_full_test))).reshape(-1)
    finally:
        os.chdir(_cwd)

    ate_hat = float(tau_hat.mean())
    runtime = time.time() - t0
    results["Do-PFN"] = dict(
        tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
        pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
    )
    print(f"✓ Do-PFN | PEHE={results['Do-PFN']['pehe']:.3f}  "
          f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")
except Exception as e:
    if "torch.nn.modules.transformer" in str(e):
        print(f"✗ Do-PFN failed: incompatible torch version ({e})\n"
              "  1. Run the '0. One-time environment check' cell at the top of this notebook.\n"
              "  2. Then RESTART THE KERNEL (Kernel/Runtime > Restart) -- if this cell already\n"
              "     failed once in this session, re-running it alone will NOT retry cleanly.\n"
              "  3. Then re-run all cells from the top.")
    else:
        print(f"✗ Do-PFN failed: {type(e).__name__}: {e}")


## 4. CausalFM

Also not on PyPI, and it additionally needs a pretrained checkpoint file:

1. `git clone https://github.com/yccm/CausalFM-toolkit.git`
2. install just the extra deps it needs (`einops`, `tabpfn==2.0.9`,
   `tensorboard`) — **not** its bundled `requirements.txt`, which is a frozen
   Linux/CUDA dev snapshot that won't install on macOS or a Colab CPU runtime
3. add the toolkit root to `sys.path`
4. point at the real checkpoint path:
   `checkpoints/checkpoints_standard/best_model.pth` (note: **not**
   `checkpoints/best_model.pth`, despite the toolkit's own README)

The toolkit's README quick-start shows
`model.estimate_cate(x_train, a_train, y_train, x_test)` with plain numpy
arrays, but `StandardCATEModel.estimate_cate` actually requires `torch.Tensor`
inputs with treatment/outcome reshaped to `[N, 1]` — the call below uses the
shapes it actually needs.

**On Colab you may see a harmless pip resolver warning here** —
`tabpfn==2.0.9` pins `huggingface-hub<1`, which conflicts with Colab's
preinstalled `gradio`/`transformers`. pip still installs everything requested;
since this notebook never imports those packages, it's safe to ignore —
CausalFM loads and runs normally right after.

In [ ]:
CAUSALFM_DIR = "CausalFM-toolkit"
CAUSALFM_URL = "https://github.com/yccm/CausalFM-toolkit.git"
CAUSALFM_CHECKPOINT = f"{CAUSALFM_DIR}/checkpoints/checkpoints_standard/best_model.pth"

if not os.path.exists(CAUSALFM_DIR):
    print(f"Cloning {CAUSALFM_URL} ...")
    subprocess.run(["git", "clone", CAUSALFM_URL], check=True)
sys.path.insert(0, os.path.abspath(CAUSALFM_DIR))

if IN_COLAB:
    get_ipython().system('pip install -q einops "tabpfn==2.0.9" tensorboard')
# Locally: uv pip install einops "tabpfn==2.0.9" tensorboard

try:
    from causalfm.models import StandardCATEModel
except ImportError:
    StandardCATEModel = None

if StandardCATEModel is None:
    print("✗ causalfm not importable -- see the install comments above.")
elif not os.path.exists(CAUSALFM_CHECKPOINT):
    print(f"✗ checkpoint not found at {CAUSALFM_CHECKPOINT}")
else:
    t0 = time.time()
    model = StandardCATEModel.from_pretrained(CAUSALFM_CHECKPOINT)

    X_train_t = torch.as_tensor(X_train, dtype=torch.float32)
    T_train_t = torch.as_tensor(T_train, dtype=torch.float32).reshape(-1, 1)
    Y_train_t = torch.as_tensor(Y_train, dtype=torch.float32).reshape(-1, 1)
    X_test_t = torch.as_tensor(X_test, dtype=torch.float32)

    result = model.estimate_cate(X_train_t, T_train_t, Y_train_t, X_test_t)
    tau_hat = result["cate"].detach().cpu().numpy().reshape(-1)

    ate_hat = float(tau_hat.mean())
    runtime = time.time() - t0
    results["CausalFM"] = dict(
        tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
        pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
    )
    print(f"✓ CausalFM | PEHE={results['CausalFM']['pehe']:.3f}  "
          f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")

## 5. Compare & visualize

Whatever subset of the three models ran successfully in your environment gets
plotted here — left panel: estimated CATE vs. ground truth (perfect
predictions sit on the diagonal); right panel: PEHE per model (lower is
better). Colors are assigned per model, consistently across both panels.

In [ ]:
import matplotlib.pyplot as plt

# Fixed model -> color assignment, consistent across every panel below.
MODEL_COLORS = {"CausalPFN": "#2a78d6", "Do-PFN": "#eb6834", "CausalFM": "#1baf7a"}

if not results:
    print("No foundation model ran successfully in this environment -- "
          "see the ✗/⚠ messages above for what to install.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: predicted vs. true CATE, one scatter series per model
    ax = axes[0]
    lo = min(tau_test.min(), *[r["tau_hat"].min() for r in results.values()])
    hi = max(tau_test.max(), *[r["tau_hat"].max() for r in results.values()])
    ax.plot([lo, hi], [lo, hi], color="#8a8a86", linewidth=1.5, linestyle="--", label="perfect (y = x)")
    for name, r in results.items():
        ax.scatter(tau_test, r["tau_hat"], s=14, alpha=0.5,
                    color=MODEL_COLORS[name], label=name)
    ax.set_xlabel("True CATE")
    ax.set_ylabel("Predicted CATE")
    ax.set_title("Predicted vs. true treatment effect")
    ax.legend(frameon=False, fontsize=9)

    # Right: PEHE bar per model (lower = better)
    ax = axes[1]
    names = list(results.keys())
    pehes = [results[n]["pehe"] for n in names]
    ax.bar(names, pehes, color=[MODEL_COLORS[n] for n in names])
    for i, v in enumerate(pehes):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("PEHE (lower is better)")
    ax.set_title("Heterogeneous-effect error, discount-email example")

    plt.tight_layout()
    plt.savefig("foundation_models_sandbox.png", dpi=150)
    plt.show()

    summary = pd.DataFrame({
        name: {"ATE_hat": r["ate_hat"], "True_ATE": true_ate, "PEHE": r["pehe"],
               "ATE_abs_error": r["ate_abs_error"], "Runtime (s)": r["runtime"]}
        for name, r in results.items()
    }).T
    print(summary.round(3))

## Reference output — verified successful run (Colab, GPU runtime)

If your numbers look like this, your setup is correct. Fixed
`SEED = 42` makes the dataset and train/test split identical every run, so
`True_ATE` will always read **1.967** — differences beyond that come from the
usual sources of nondeterminism in the models themselves (GPU non-determinism,
library version drift, CPU vs. GPU execution), not from your setup being wrong.

| Model | ATE_hat | True_ATE | PEHE | ATE_abs_error | Runtime (s) |
|---|---|---|---|---|---|
| CausalPFN | 1.911 | 1.967 | 0.237 | 0.056 | 10.059 |
| Do-PFN | 1.652 | 1.967 | 1.407 | 0.314 | 8.264 |
| CausalFM | 0.646 | 1.967 | 2.194 | 1.320 | 0.311 |

![Reference plot: predicted vs. true CATE, and PEHE per model](assets/reference_output_colab.png)

**CausalPFN** (blue) tracks the true CATE closely across its full range,
hugging the `y = x` diagonal. **Do-PFN** (orange) gets the direction right but
compresses the range, under-predicting the largest true effects. **CausalFM**
(green) clusters its predictions in a narrow band regardless of the true
effect size, which is why it posts the highest PEHE here — a reminder that a
low `ATE_abs_error` (CausalPFN: 0.056) doesn't by itself guarantee
well-calibrated *heterogeneous* (CATE-level) predictions, and vice versa.

## Key takeaways for practitioners

- **Same idea, different APIs.** All three are in-context learners, but no
  shared method signature — read each library's own quick-start rather than
  assuming one model's calling convention carries over to another. Or use this
  repo's `causal_bench` wrappers to get a unified signature across all three
  (see [`Lalonde_benchmark.ipynb`](Lalonde_benchmark.ipynb)).
- **Install cost varies.** CausalPFN is a plain `pip install`; Do-PFN and
  CausalFM need `git clone` + `sys.path` wiring, and CausalFM also needs a
  checkpoint file — a one-time cost per environment.
- **"Fit" isn't training.** No hyperparameters, no train/val split — your data
  is just context for a frozen network. A bad prediction means bad input data,
  not a tuning problem.
- **The naive comparison in §1 is biased on purpose** — that's the whole
  reason to use a causal method at all, foundation model or classic
  metalearner.
- **Ground truth is a simulation-only luxury.** PEHE only works here because
  `tau_true` is known. On real data, we need to have other metrics like sensitivity gain and qini.
- **CausalPFN's Apple Silicon issue is a platform kernel bug, not a CUDA
  requirement** — it runs fine on Colab either way. Do-PFN and CausalFM ran
  fine on CPU; expect it to just be slower without a GPU.